## Sentiment analysis 

In [ ]:
##Generating Sentiment Scores for News Headlines

In [2]:
import os
import time
import logging
import torch
import torch.nn.functional as F
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from dateutil import parser

# =====================================================
LOG_FILE = "processing_log.txt"
logging.basicConfig(
    filename=LOG_FILE,
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

def log(msg, level="info"):
    tqdm.write(msg)
    if level == "error":
        logging.error(msg)
        
    elif level == "warning":
        logging.warning(msg)

    else:
        logging.info(msg)

def normalize_date_column(df):
    def parse_date_safe(x):
        try:
            return parser.parse(str(x), dayfirst=False)

        except Exception:
            try:
                return parser.parse(str(x), dayfirst=True)
            except Exception:
                return None

    df["Date"] = df["Date"].apply(parse_date_safe)
    df = df.dropna(subset=["Date"])
    df = df.sort_values("Date").reset_index(drop=True)
    return df

def split_headlines(text):
    if pd.isna(text):
        return []
    return [t.strip() for t in str(text).split('|') if t.strip()]

# =====================================================

DATA_DIR = "/home/sunkari/Stock_price_predictor/Dataset" 
OUTPUT_DIR = "/home/sunkari/Stock_price_predictor/Processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =====================================================

MODEL_NAME = "yiyanghkust/finbert-tone"
log(f"🔹 Loading model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.eval()
log("✅ Model loaded successfully.")

# =====================================================

def get_sentiment_scores(texts):

    if len(texts) == 0:
        return [0.0, 1.0, 0.0]  

    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=-1)
    return probs.mean(dim=0).numpy().tolist()
# =====================================================

start_time_total = time.time()
for file in os.listdir(DATA_DIR):

    if not file.endswith(".csv"):
        continue

    start_time = time.time()
    company_path = os.path.join(DATA_DIR, file)
    log(f"\n🔍 Processing {file} ...")

    try:
        df = pd.read_csv(company_path)
        df.columns = df.columns.str.strip()
        log(f"📂 Loaded file with {len(df)} rows.")

        df = normalize_date_column(df)
        log(f"🗓️ Normalized dates, {len(df)} rows remain after cleaning.")

        # Split headlines into lists
        df["Headline_List"] = df["Headlines"].apply(split_headlines)

        # Compute daily sentiment
        sentiments = []

        for headlines in tqdm(df["Headline_List"].tolist(), desc=f"Sentiment {file}"):
            try:
                probs = get_sentiment_scores(headlines)
                sentiments.append(probs)

            except Exception as e:

                log(f"⚠️ Error processing headlines: {headlines[:3]}... | {e}", "warning")
                sentiments.append([0.0, 1.0, 0.0])  # default neutral

        sentiments = pd.DataFrame(sentiments, columns=["negative", "neutral", "positive"])

        df = pd.concat([df, sentiments], axis=1)

        # Drop the text columns after processing
        df = df.drop(columns=["Headlines", "Headline_List"], errors='ignore')
        
        # Save processed versions
        base_name = os.path.splitext(file)[0]
        train_out = os.path.join(OUTPUT_DIR, f"{base_name}_train.csv")
        df.to_csv(train_out, index=False)
        elapsed = time.time() - start_time
        log(f"✅ Saved data split for {file} | data={len(df)} | ⏱️ {elapsed:.2f}s")

    except Exception as e:

        log(f"❌ Error processing {file}: {e}", "error")

total_time = time.time() - start_time_total

log(f"\n🏁 All files processed successfully in {total_time/60:.2f} minutes.")

🔹 Loading model: yiyanghkust/finbert-tone
✅ Model loaded successfully.

🔍 Processing XOM_stock_gdelt_final.csv ...
📂 Loaded file with 1003 rows.
🗓️ Normalized dates, 1003 rows remain after cleaning.


Sentiment XOM_stock_gdelt_final.csv: 100%|██████████| 1003/1003 [01:24<00:00, 11.88it/s]


✅ Saved data split for XOM_stock_gdelt_final.csv | data=1003 | ⏱️ 84.51s

🔍 Processing MSFT_stock_gdelt_final.csv ...
📂 Loaded file with 1003 rows.
🗓️ Normalized dates, 1003 rows remain after cleaning.


Sentiment MSFT_stock_gdelt_final.csv: 100%|██████████| 1003/1003 [02:01<00:00,  8.27it/s]


✅ Saved data split for MSFT_stock_gdelt_final.csv | data=1003 | ⏱️ 121.43s

🔍 Processing V_stock_gdelt_final.csv ...
📂 Loaded file with 1003 rows.
🗓️ Normalized dates, 1003 rows remain after cleaning.


...

✅ Saved data split for TSLA_stock_gdelt_final.csv | data=1003 | ⏱️ 137.65s

🔍 Processing JPM_stock_gdelt_final.csv ...
📂 Loaded file with 1003 rows.
🗓️ Normalized dates, 1003 rows remain after cleaning.


Sentiment JPM_stock_gdelt_final.csv: 100%|██████████| 1003/1003 [02:07<00:00,  7.85it/s]


✅ Saved data split for JPM_stock_gdelt_final.csv | data=1003 | ⏱️ 127.95s

🔍 Processing AAPL_stock_gdelt_final.csv ...
📂 Loaded file with 1003 rows.
🗓️ Normalized dates, 1003 rows remain after cleaning.


Sentiment AAPL_stock_gdelt_final.csv: 100%|██████████| 1003/1003 [01:45<00:00,  9.48it/s]

✅ Saved data split for AAPL_stock_gdelt_final.csv | data=1003 | ⏱️ 105.91s

🏁 All files processed successfully in 18.89 minutes.


In [ ]:
##Preparing Data Windows for Transformer Model

In [3]:
import os
import glob
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import pickle
from tqdm import tqdm
import random

# =====================================================
# CONFIGURATION
# =====================================================
INPUT_DIR = "/home/sunkari/Stock_price_predictor/Processed"
OUTPUT_DIR = "windows_new"
SCALER_DIR = "scalers_new"
WINDOW_SIZE = 8
TARGET_COLUMN = "Close"
TEST_SPLIT_RATIO = 0.2

NUMERIC_COLS = [
    "Adj Close", "Close", "High", "Low", "Open",
    "Volume", "Daily_Return", "EMA_7", "EMA_21"
]
SENTIMENT_COLS = ["negative", "neutral", "positive"]

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SCALER_DIR, exist_ok=True)

print("📂 Building train/test windows including next-day sentiment columns...")

# -----------------------------
# SELECT COMPANIES
# -----------------------------
all_files = glob.glob(os.path.join(INPUT_DIR, "*.csv"))
all_companies = [os.path.splitext(os.path.basename(f))[0] for f in all_files]

# Pick first 10 companies as an example
train_companies = all_companies[:int(len(all_companies) * (1 - TEST_SPLIT_RATIO))]
test_companies = all_companies[int(len(all_companies) * (1 - TEST_SPLIT_RATIO)):]

train_data, test_data = [], []
train_company_list, test_company_list = [], []

# -----------------------------
# LOOP THROUGH FILES
# -----------------------------
for file_path in tqdm(all_files):
    company_name = os.path.splitext(os.path.basename(file_path))[0]

    if company_name not in train_companies + test_companies:
        continue  # skip companies not selected

    df = pd.read_csv(file_path).dropna(subset=[TARGET_COLUMN]).reset_index(drop=True)

    if not all(col in df.columns for col in NUMERIC_COLS + SENTIMENT_COLS):
        print(f"⚠️ Skipping {company_name} — missing columns.")
        continue


    scaler = MinMaxScaler()
    scaler.fit(df[NUMERIC_COLS])
    with open(os.path.join(SCALER_DIR, f"{company_name}_scaler.pkl"), "wb") as f:
        pickle.dump(scaler, f)

    # Inside build_windows function
    def build_windows(subset, company_name, scaler):
        numeric_scaled = scaler.transform(subset[NUMERIC_COLS])
        sentiment_values = subset[SENTIMENT_COLS].values
        numeric_values = subset[NUMERIC_COLS].values
        combined = np.concatenate([numeric_scaled, sentiment_values], axis=1)
        data = []

        # Scale target y using the same MinMaxScaler
        target_column_idx = NUMERIC_COLS.index(TARGET_COLUMN)
        y_values = numeric_values[:, target_column_idx].reshape(-1, 1)
        y_scaler = MinMaxScaler()
        y_scaled = y_scaler.fit_transform(y_values)

        # Save y scaler for this company
        with open(os.path.join(SCALER_DIR, f"{company_name}_y_scaler.pkl"), "wb") as f:
            pickle.dump(y_scaler, f)

        for i in range(len(combined) - WINDOW_SIZE):
            X_seq = combined[i:i + WINDOW_SIZE]
            if i + WINDOW_SIZE >= len(sentiment_values):
                continue
            next_sentiment = sentiment_values[i + WINDOW_SIZE]
            y = y_scaled[i + WINDOW_SIZE][0]  # scaled target
            data.append((X_seq, next_sentiment, y, company_name))

        return data



    # -----------------------------
    # ADD TO TRAIN OR TEST
    # -----------------------------
    if company_name in train_companies:
        train_data.extend(build_windows(df, company_name, scaler))
        train_company_list.append(company_name)
    else:  # test_companies
        test_data.extend(build_windows(df, company_name, scaler))
        test_company_list.append(company_name)

# -----------------------------
# SHUFFLE DATA
# -----------------------------
random.shuffle(train_data)
random.shuffle(test_data)
# -----------------------------
# SAVE PICKLES
# -----------------------------
pickle.dump(train_data, open(os.path.join(OUTPUT_DIR, "train_windows.pkl"), "wb"))
pickle.dump(test_data, open(os.path.join(OUTPUT_DIR, "test_windows.pkl"), "wb"))
pickle.dump(train_company_list, open(os.path.join(OUTPUT_DIR, "train_company_list.pkl"), "wb"))
pickle.dump(test_company_list, open(os.path.join(OUTPUT_DIR, "test_company_list.pkl"), "wb"))
print(f"✅ Train samples: {len(train_data)} | Companies: {train_company_list}")
print(f"✅ Test samples: {len(test_data)} | Companies: {test_company_list}")
print("\n✅ Preprocessing complete.")

📂 Building train/test windows including next-day sentiment columns...


100%|██████████| 10/10 [00:00<00:00, 48.26it/s]


✅ Train samples: 7960 | Companies: ['XOM_stock_gdelt_final_train', 'NVDA_stock_gdelt_final_train', 'V_stock_gdelt_final_train', 'TSLA_stock_gdelt_final_train', 'GOOG_stock_gdelt_final_train', 'JPM_stock_gdelt_final_train', 'PFE_stock_gdelt_final_train', 'AMZN_stock_gdelt_final_train']
✅ Test samples: 1990 | Companies: ['MSFT_stock_gdelt_final_train', 'AAPL_stock_gdelt_final_train']

✅ Preprocessing complete.


In [ ]:
import os
import glob
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import pickle
from tqdm import tqdm
import random

# =====================================================
# CONFIGURATION
# =====================================================
INPUT_DIR = "/home/sunkari/Stock_price_predictor/Processed"
OUTPUT_DIR = "windows_b"
SCALER_DIR = "scaler_b"
WINDOW_SIZE = 8
TRAIN_RATIO = 0.8
TARGET_COLUMN = "Close"

NUMERIC_COLS = [
    "Adj Close", "Close", "High", "Low", "Open",
    "Volume", "Daily_Return", "EMA_7", "EMA_21"
]

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SCALER_DIR, exist_ok=True)

print("📂 Building windows for all CSVs...")

all_windows = []

# -----------------------------
# LOOP THROUGH FILES
# -----------------------------
all_files = glob.glob(os.path.join(INPUT_DIR, "*.csv"))

for file_path in tqdm(all_files):
    company_name = os.path.splitext(os.path.basename(file_path))[0]
    df = pd.read_csv(file_path).dropna(subset=[TARGET_COLUMN]).reset_index(drop=True)

    if not all(col in df.columns for col in NUMERIC_COLS):
        print(f"⚠️ Skipping {company_name} — missing columns.")
        continue

    # -----------------------------
    # SCALE FEATURES
    # -----------------------------
    scaler = MinMaxScaler()
    numeric_scaled = scaler.fit_transform(df[NUMERIC_COLS])

    # Save scaler for later denormalization
    scaler_path = os.path.join(SCALER_DIR, f"{company_name}_scaler.pkl")
    with open(scaler_path, "wb") as f:
        pickle.dump(scaler, f)
    # Now each company has its own scaler file

    # -----------------------------
    # BUILD WINDOWS
    # -----------------------------
    target_idx = NUMERIC_COLS.index(TARGET_COLUMN)
    y_values = df[TARGET_COLUMN].values

    for i in range(len(df) - WINDOW_SIZE):
        X_seq = numeric_scaled[i:i + WINDOW_SIZE]  # last 8 days
        y = y_values[i + WINDOW_SIZE]             # next-day Close
        all_windows.append((X_seq, y))

# -----------------------------
# SHUFFLE ALL WINDOWS
# -----------------------------
random.shuffle(all_windows)

# -----------------------------
# SPLIT TRAIN / TEST
# -----------------------------
num_train = int(len(all_windows) * TRAIN_RATIO)
train_data = all_windows[:num_train]
test_data = all_windows[num_train:]

# -----------------------------
# SAVE PICKLES
# -----------------------------
pickle.dump(train_data, open(os.path.join(OUTPUT_DIR, "train_windows.pkl"), "wb"))
pickle.dump(test_data, open(os.path.join(OUTPUT_DIR, "test_windows.pkl"), "wb"))

print(f"✅ Total windows: {len(all_windows)}")
print(f"✅ Train windows: {len(train_data)}")
print(f"✅ Test windows: {len(test_data)}")
print(f"✅ Scalers saved in '{SCALER_DIR}'")
print("\n✅ Preprocessing complete.")

📂 Building windows for all CSVs...


100%|██████████| 10/10 [00:00<00:00, 42.91it/s]


✅ Total windows: 9950
✅ Train windows: 7960
✅ Test windows: 1990
✅ Scalers saved in 'scaler_b'

✅ Preprocessing complete.
